###  1: Install libraries

In [1]:
!pip install pymupdf sentence-transformers faiss-cpu numpy tqdm

### 2: Import libraries and set folder path

In [2]:
import os
import fitz
import pickle
import faiss
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

PDF_FOLDER = r"D:\RAGChat_Project\pdfs"
VECTOR_FOLDER = r"D:\RAGChat_Project\vector_store"

INDEX_FILE = os.path.join(VECTOR_FOLDER, "faiss.index")
METADATA_FILE = os.path.join(VECTOR_FOLDER, "metadata.pkl")

os.makedirs(VECTOR_FOLDER, exist_ok=True)

### 3: Define PDF text extraction function

In [3]:
def extract_text_from_pdf(pdf_path):
    pages_data = []

    try:
        pdf_document = fitz.open(pdf_path)
    except Exception as error:
        print("Could not open PDF:", pdf_path)
        print("Error:", error)
        return pages_data

    for page_number in range(len(pdf_document)):
        try:
            page = pdf_document[page_number]
            text = page.get_text("text")

            text = text.replace("\n", " ")
            text = " ".join(text.split())

            if len(text) > 50:
                pages_data.append({
                    "page_number": page_number + 1,
                    "text": text
                })

        except Exception as error:
            print("Skipped page:", page_number + 1)
            print("Reason:", error)

    return pages_data

### 4: Define chunking function

In [4]:
def make_chunks(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if len(chunk) > 100:
            chunks.append(chunk)

        start = start + chunk_size - overlap

    return chunks

### 5: Read all PDFs and create chunks

In [5]:
all_chunks = []
metadata = []

pdf_files = [
    file for file in os.listdir(PDF_FOLDER)
    if file.lower().endswith(".pdf")
]

print("Total PDF files found:", len(pdf_files))

if len(pdf_files) == 0:
    print("No PDF files found. Please check your PDF folder path.")
else:
    for pdf_file in tqdm(pdf_files):
        pdf_path = os.path.join(PDF_FOLDER, pdf_file)

        pages = extract_text_from_pdf(pdf_path)

        for page in pages:
            chunks = make_chunks(page["text"])

            for chunk in chunks:
                all_chunks.append(chunk)

                metadata.append({
                    "filename": pdf_file,
                    "page_number": page["page_number"],
                    "text": chunk
                })

    print("Total chunks created:", len(all_chunks))

Total PDF files found: 10


 80%|████████  | 8/10 [00:17<00:03,  1.90s/it]

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: uns

100%|██████████| 10/10 [00:22<00:00,  2.30s/it]

Total chunks created: 6434


### 6: Create embeddings and FAISS vector store

In [6]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    all_chunks,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype("float32"))

faiss.write_index(index, INDEX_FILE)

with open(METADATA_FILE, "wb") as file:
    pickle.dump(metadata, file)

print("Vector database created and saved.")
print("Total vectors stored:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/202 [00:00<?, ?it/s]

Vector database created and saved.
Total vectors stored: 6434


### 7: Define search function

In [7]:
def search_pdf(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(question_embedding, top_k)

    results = []

    for score, chunk_id in zip(scores[0], ids[0]):
        if chunk_id == -1:
            continue

        item = metadata[chunk_id]

        results.append({
            "score": float(score),
            "filename": item["filename"],
            "page_number": item["page_number"],
            "text": item["text"]
        })

    return results

### 8: Ask question

In [11]:
question = "What is a Neuronal Signaling?"

results = search_pdf(question, top_k=3)

print("Question:", question)

for i, result in enumerate(results, start=1):
    print("\nResult", i)
    print("PDF:", result["filename"])
    print("Page:", result["page_number"])
    print("Score:", round(result["score"], 3))
    print("Text:", result["text"][:800])
    print("-" * 80)

Question: What is a Neuronal Signaling?

Result 1
PDF: 1. Computational.Neuroscience-A.Comprehensive.Approach.pdf
Page: 143
Score: 0.63
Text: have sensible interpretations in terms of neuronal signalling to blood vessels. ‡A form of signalling which might also be useful in an artiﬁcial neural network (see Section 4.4). © 2004 by Chapman & Hall/CRC
--------------------------------------------------------------------------------

Result 2
PDF: 3. J Keener physiology.pdf
Page: 196
Score: 0.602
Text: C H A P T E R 4 Passive Electrical Flow in Neurons Neurons are among the most important and interesting cells in the body. They are the fundamental building blocks of the central nervous system and hence responsible for motor control, cognition, perception, and memory, among other things. Although our understanding of how networks of neurons interact to form an intelligent system is extremely limited, one prerequisite for an understanding of the nervous system is an understanding of how indivi